# Final Model Evaluation Against Manual Audit Labels

This notebook documents the evaluation of the five final models against a completed stratified manual audit from the locked-test partition. It preserves the separation between silver-label model development and post-prediction manual-label assessment.


##  Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, completed manual-audit records, row-level predictions, bootstrap replicates, or private run artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the restricted manual-audit evaluation contract

Imports metric utilities, defines the restricted project-root placeholder, identifies the completed audit, locked-test, and repeated-seed prediction artifacts, and specifies the expected audit design, seeds, and five final models.


In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_label(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    return value if value in VALID_LABELS else np.nan


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
MANUAL_AUDIT_DIR = ROUND1_DIR / 'data' / 'manual_audit_locked_test'
REPORT_DIR = ROUND1_DIR / 'reports' / 'human_label_audit'
FINAL_EVALUATION_REPORT_DIR = REPORT_DIR / 'r1_08_repeated_seed'
FIGURE_DIR = ROUND1_DIR / 'figures' / 'human_label_audit' / 'r1_08_repeated_seed'

RECONCILED_AUDIT_PATH = MANUAL_AUDIT_DIR / 'locked_test_audit_completed_300_private.csv'
AGREEMENT_SUMMARY_PATH = REPORT_DIR / 'all_300_annotation_agreement.json'
RECONCILIATION_SUMMARY_PATH = REPORT_DIR / 'all_300_annotation_reconciliation_summary.json'
LOCKED_TEST_PATH = PROJECT_ROOT / 'data' / 'final_test_locked' / 'test_locked.csv'

R1_08_ROOT = ROUND1_DIR / 'artifacts' / 'r1_08_repeated_seed_evaluation_v1'
R1_08_PREDICTIONS_PATH = R1_08_ROOT / 'locked_test_predictions_per_model_long_with_silver.csv'
R1_08_MANIFEST_PATH = R1_08_ROOT / 'run_manifest.json'
R1_08_STATE_PATH = R1_08_ROOT / 'run_state.json'
R1_08_CONTRACT_PATH = R1_08_ROOT / 'prediction_contract.json'

LABEL_ORDER = ['negatif', 'netral', 'positif']
VALID_LABELS = set(LABEL_ORDER)
EXPECTED_SEEDS = [42, 52, 62]
EXPECTED_AUDIT_SIZE = 300
TARGET_PER_SILVER_CLASS = 100
FINAL_MODEL_ORDER = [
    ('svm', 'SVM'),
    ('indobert', 'IndoBERT'),
    ('wa_svm_bert', 'WA (SVM+BERT)'),
    ('xgboost', 'XGBoost'),
    ('calibrated_xgboost', 'Calibrated XGBoost'),
]

for required_path in [
    RECONCILED_AUDIT_PATH, AGREEMENT_SUMMARY_PATH, RECONCILIATION_SUMMARY_PATH, LOCKED_TEST_PATH,
    R1_08_PREDICTIONS_PATH, R1_08_MANIFEST_PATH, R1_08_STATE_PATH, R1_08_CONTRACT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required input was not found: {required_path}')



print('Reconciled audit:', RECONCILED_AUDIT_PATH)
print('Evaluation output directory:', FINAL_EVALUATION_REPORT_DIR)


### Validate the restricted manual-audit inputs

Loads the restricted 300-review audit only within the authorized analysis environment. It verifies sampling strata, adjudication consistency, agreement, and alignment with locked-test silver labels. No audit record, annotator decision, or text field is written or displayed by this release copy.

In [ ]:
human_audit = pd.read_csv(RECONCILED_AUDIT_PATH)
locked_test_df = pd.read_csv(LOCKED_TEST_PATH)
agreement_summary = json.loads(AGREEMENT_SUMMARY_PATH.read_text(encoding='utf-8'))
reconciliation_summary = json.loads(RECONCILIATION_SUMMARY_PATH.read_text(encoding='utf-8'))

required_audit_columns = {
    'locked_test_row', 'silver_label_lstm', 'final_manual_label',
    'annotator_1_label', 'annotator_2_label', 'consensus_label', 'needs_adjudication', 'design_weight'
}
missing_audit_columns = sorted(required_audit_columns.difference(human_audit.columns))
if missing_audit_columns:
    raise ValueError(f'Reconciled audit is missing columns: {missing_audit_columns}')
if len(human_audit) != EXPECTED_AUDIT_SIZE:
    raise RuntimeError(f'Reconciled audit must contain {EXPECTED_AUDIT_SIZE} reviews, found {len(human_audit)}.')
if human_audit['locked_test_row'].duplicated().any():
    raise RuntimeError('The reconciled audit contains duplicate locked-test row references.')

for column in ['silver_label_lstm', 'final_manual_label', 'annotator_1_label', 'annotator_2_label', 'consensus_label']:
    human_audit[column] = human_audit[column].map(normalize_label)
if human_audit[['silver_label_lstm', 'final_manual_label', 'annotator_1_label', 'annotator_2_label']].isna().any().any():
    raise RuntimeError('The reconciled audit has invalid silver, final manual, or annotator labels.')

recomputed_needs_adjudication = human_audit['annotator_1_label'] != human_audit['annotator_2_label']
stored_needs_adjudication = human_audit['needs_adjudication'].astype(str).str.strip().str.lower().eq('true')
if not recomputed_needs_adjudication.eq(stored_needs_adjudication).all():
    raise RuntimeError('Stored needs_adjudication does not match the two annotator labels.')
if human_audit.loc[recomputed_needs_adjudication, 'consensus_label'].isna().any():
    raise RuntimeError('At least one annotator disagreement has no consensus label.')

reconstructed_final_label = human_audit['annotator_1_label'].copy()
reconstructed_final_label.loc[recomputed_needs_adjudication] = human_audit.loc[
    recomputed_needs_adjudication, 'consensus_label'
]
if not reconstructed_final_label.eq(human_audit['final_manual_label']).all():
    raise RuntimeError('Final manual labels do not match the reconstruction from annotators and consensus.')


silver_counts = human_audit['silver_label_lstm'].value_counts().reindex(LABEL_ORDER, fill_value=0)
if not silver_counts.eq(TARGET_PER_SILVER_CLASS).all():
    raise RuntimeError(f'The audit must contain 100 reviews per silver-label class: {silver_counts.to_dict()}')
if human_audit['design_weight'].isna().any():
    raise RuntimeError('The reconciled audit has missing design weights.')

agreement = float((~recomputed_needs_adjudication).mean())
kappa = float(cohen_kappa_score(human_audit['annotator_1_label'], human_audit['annotator_2_label'], labels=LABEL_ORDER))
if not np.isclose(agreement, agreement_summary['percent_agreement_before_adjudication']):
    raise RuntimeError('Agreement does not match the final reconciliation summary.')
if not np.isclose(kappa, agreement_summary['cohen_kappa_before_adjudication']):
    raise RuntimeError('Cohen kappa does not match the final reconciliation summary.')
if reconciliation_summary.get('all_reconstructed_final_labels_match_current_final_audit') is not True:
    raise RuntimeError('The audit reconciliation did not verify final-label identity.')

locked_labels = locked_test_df['sentiment_label'].map(normalize_label)
locked_label_by_row = pd.Series(locked_labels.to_numpy(), index=np.arange(len(locked_labels), dtype=np.int64))
audit_source_labels = human_audit['locked_test_row'].astype(np.int64).map(locked_label_by_row)
if not audit_source_labels.eq(human_audit['silver_label_lstm']).all():
    raise RuntimeError('Audit silver labels do not match the locked-test source rows.')

print('Audit rows:', len(human_audit))
print('Agreement before adjudication:', f'{agreement:.4%}')
print('Cohen kappa before adjudication:', f'{kappa:.4f}')
print('Cases requiring consensus:', int(recomputed_needs_adjudication.sum()))


### Validate repeated-seed predictions before label attachment

Checks that the repeated-seed evaluation is complete, its locked-test predictions were produced label-blind, seed and model contracts are stable, checksums match, and silver labels align with the locked-test source.


In [ ]:
r1_08_manifest = json.loads(R1_08_MANIFEST_PATH.read_text(encoding='utf-8'))
r1_08_state = json.loads(R1_08_STATE_PATH.read_text(encoding='utf-8'))
r1_08_contract = json.loads(R1_08_CONTRACT_PATH.read_text(encoding='utf-8'))
if r1_08_state.get('status') != 'completed' or r1_08_manifest.get('locked_test_metrics_computed') is not True:
    raise RuntimeError('Repeated-seed evaluation is incomplete or does not contain valid locked-test artifacts.')
evaluation_target = str(r1_08_manifest.get('evaluation_target', '')).lower()
if 'lstm-derived silver labels' not in evaluation_target or 'locked test set' not in evaluation_target:
    raise RuntimeError('The repeated-seed evaluation target does not describe silver-label evaluation on the locked test set.')
if r1_08_contract.get('manual_labels_included') is not False:
    raise RuntimeError('The repeated-seed prediction contract must not include manual labels.')
if r1_08_contract.get('seeds') != EXPECTED_SEEDS:
    raise RuntimeError('Unexpected repeated-seed evaluation seeds: ' + str(r1_08_contract.get('seeds')))

expected_prediction_hash = r1_08_manifest.get('output_checksums', {}).get(R1_08_PREDICTIONS_PATH.name)
if expected_prediction_hash != sha256_file(R1_08_PREDICTIONS_PATH):
    raise RuntimeError('The repeated-seed prediction file checksum does not match its manifest.')
predictions_long = pd.read_csv(R1_08_PREDICTIONS_PATH)

required_prediction_columns = {
    'seed', 'row_id', 'y_silver', 'y_silver_label', 'model_key', 'model',
    'prob_negatif', 'prob_netral', 'prob_positif', 'prediction', 'prediction_label', 'confidence', 'entropy',
}
missing_prediction_columns = sorted(required_prediction_columns.difference(predictions_long.columns))
if missing_prediction_columns:
    raise ValueError(f'Repeated-seed predictions are missing columns: {missing_prediction_columns}')
if sorted(predictions_long['seed'].unique().tolist()) != EXPECTED_SEEDS:
    raise RuntimeError('The prediction file does not contain exactly the expected three seeds.')
if set(predictions_long['model_key']) != {model_key for model_key, _ in FINAL_MODEL_ORDER}:
    raise RuntimeError('The prediction file does not contain exactly the five final models.')
if len(predictions_long) != len(locked_test_df) * len(EXPECTED_SEEDS) * len(FINAL_MODEL_ORDER):
    raise RuntimeError('Prediction count does not equal seeds x locked-test reviews x final models.')
if predictions_long.duplicated(['seed', 'row_id', 'model_key']).any():
    raise RuntimeError('Prediction keys are duplicated.')

expected_keys = {
    (seed, row_id, model_key)
    for seed in EXPECTED_SEEDS
    for row_id in range(len(locked_test_df))
    for model_key, _ in FINAL_MODEL_ORDER
}
observed_keys = set(zip(
    predictions_long['seed'].astype(int), predictions_long['row_id'].astype(int), predictions_long['model_key']
))
if observed_keys != expected_keys:
    raise RuntimeError('Prediction keys do not cover the complete seed, row, and model contract.')

locked_reference = pd.DataFrame({
    'row_id': np.arange(len(locked_test_df), dtype=np.int64),
    'source_y_silver_label': locked_labels.to_numpy(),
    'source_y_silver': locked_labels.map({label: index for index, label in enumerate(LABEL_ORDER)}).to_numpy(dtype=np.int64),
})
prediction_silver_check = predictions_long.merge(locked_reference, on='row_id', how='left', validate='many_to_one')
if not prediction_silver_check['y_silver'].eq(prediction_silver_check['source_y_silver']).all():
    raise RuntimeError('Repeated-seed y_silver does not match the locked-test source.')
if not prediction_silver_check['y_silver_label'].eq(prediction_silver_check['source_y_silver_label']).all():
    raise RuntimeError('Repeated-seed y_silver_label does not match the locked-test source.')
if not predictions_long['prediction_label'].map(normalize_label).notna().all():
    raise RuntimeError('Repeated-seed evaluation contains a prediction label outside the three audit classes.')

print('Prediction rows:', len(predictions_long))
print('Seeds:', EXPECTED_SEEDS)


### Calculate aggregate final-model metrics against manual labels

Temporarily links seed-specific predictions to final manual labels to calculate aggregate unweighted and design-weighted metrics, silver-to-manual alignment, and confusion counts. The intermediate row-level evaluation frame is retained only in memory and is never exported.

In [ ]:
audit_for_evaluation = human_audit[[
    'locked_test_row', 'silver_label_lstm', 'final_manual_label', 'design_weight',
]].copy().rename(columns={'locked_test_row': 'row_id'})
audit_for_evaluation['row_id'] = audit_for_evaluation['row_id'].astype(np.int64)

evaluation_long = predictions_long.merge(
    audit_for_evaluation, on='row_id', how='inner', validate='many_to_one'
)
expected_evaluation_rows = len(audit_for_evaluation) * len(EXPECTED_SEEDS) * len(FINAL_MODEL_ORDER)
if len(evaluation_long) != expected_evaluation_rows:
    raise RuntimeError('Some audit reviews are missing predictions for a seed or final model.')
if not evaluation_long['y_silver_label'].eq(evaluation_long['silver_label_lstm']).all():
    raise RuntimeError('Repeated-seed silver labels do not match the audit strata.')

metric_rows = []
confusion_rows = []
for seed in EXPECTED_SEEDS:
    for model_key, model_name in FINAL_MODEL_ORDER:
        group = evaluation_long.loc[
            evaluation_long['seed'].eq(seed) & evaluation_long['model_key'].eq(model_key)
        ].sort_values('row_id')
        if len(group) != len(audit_for_evaluation):
            raise RuntimeError(f'Seed {seed}/{model_key} does not contain the complete audit.')
        y_manual = group['final_manual_label'].map(normalize_label)
        y_pred = group['prediction_label'].map(normalize_label)
        if y_manual.isna().any() or y_pred.isna().any():
            raise RuntimeError(f'Invalid manual or prediction label in seed {seed}/{model_key}.')
        weights = group['design_weight'].to_numpy(dtype=float)
        metric_rows.append({
            'seed': seed,
            'model_key': model_key,
            'model': model_name,
            'n_audit': int(len(group)),
            'accuracy_unweighted': float(accuracy_score(y_manual, y_pred)),
            'macro_f1_unweighted': float(f1_score(y_manual, y_pred, labels=LABEL_ORDER, average='macro', zero_division=0)),
            'balanced_accuracy_unweighted': float(balanced_accuracy_score(y_manual, y_pred)),
            'accuracy_design_weighted': float(accuracy_score(y_manual, y_pred, sample_weight=weights)),
            'macro_f1_design_weighted': float(f1_score(
                y_manual, y_pred, labels=LABEL_ORDER, average='macro', sample_weight=weights, zero_division=0
            )),
        })
        matrix = confusion_matrix(y_manual, y_pred, labels=LABEL_ORDER)
        for true_index, true_label in enumerate(LABEL_ORDER):
            for pred_index, predicted_label in enumerate(LABEL_ORDER):
                confusion_rows.append({
                    'seed': seed,
                    'model_key': model_key,
                    'model': model_name,
                    'true_manual_label': true_label,
                    'predicted_label': predicted_label,
                    'n_reviews': int(matrix[true_index, pred_index]),
                })

model_metrics_per_seed = pd.DataFrame(metric_rows).sort_values(['model_key', 'seed']).reset_index(drop=True)
metric_columns = [
    'accuracy_unweighted', 'macro_f1_unweighted', 'balanced_accuracy_unweighted',
    'accuracy_design_weighted', 'macro_f1_design_weighted',
]
summary_rows = []
for model_key, model_name in FINAL_MODEL_ORDER:
    group = model_metrics_per_seed.loc[model_metrics_per_seed['model_key'].eq(model_key)]
    row = {
        'model_key': model_key,
        'model': model_name,
        'n_seeds': int(group['seed'].nunique()),
        'seeds': ','.join(str(seed) for seed in sorted(group['seed'].unique())),
        'n_audit_per_seed': int(group['n_audit'].iloc[0]),
    }
    for metric_name in metric_columns:
        values = group[metric_name].to_numpy(dtype=float)
        row[f'{metric_name}_mean'] = float(values.mean())
        row[f'{metric_name}_std'] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
        row[f'{metric_name}_min'] = float(values.min())
        row[f'{metric_name}_max'] = float(values.max())
    summary_rows.append(row)
model_metrics_summary = pd.DataFrame(summary_rows)

base_model_evaluation = evaluation_long.loc[
    evaluation_long['model_key'].isin(['svm', 'indobert']),
    ['seed', 'row_id', 'final_manual_label', 'model_key', 'prediction_label'],
].copy()
base_model_wide = base_model_evaluation.pivot(
    index=['seed', 'row_id', 'final_manual_label'], columns='model_key', values='prediction_label'
).reset_index()
if base_model_wide[['svm', 'indobert']].isna().any().any():
    raise RuntimeError('A base-model prediction is missing from the aggregate manual-label evaluation.')

manual_complementarity_rows = []
for seed in EXPECTED_SEEDS:
    group = base_model_wide.loc[base_model_wide['seed'].eq(seed)]
    if len(group) != EXPECTED_AUDIT_SIZE:
        raise RuntimeError(f'Seed {seed} does not contain the complete manual audit for base-model complementarity.')
    svm_correct = group['svm'].eq(group['final_manual_label']).to_numpy(dtype=bool)
    indobert_correct = group['indobert'].eq(group['final_manual_label']).to_numpy(dtype=bool)
    svm_error = ~svm_correct
    indobert_error = ~indobert_correct
    if svm_error.std(ddof=0) == 0 or indobert_error.std(ddof=0) == 0:
        phi_error_correlation = float('nan')
    else:
        phi_error_correlation = float(np.corrcoef(svm_error.astype(float), indobert_error.astype(float))[0, 1])
    manual_complementarity_rows.append({
        'reference_target': 'human_audit_manual',
        'seed': int(seed),
        'n': int(len(group)),
        'both_correct_n': int(np.sum(svm_correct & indobert_correct)),
        'svm_only_correct_n': int(np.sum(svm_correct & ~indobert_correct)),
        'indobert_only_correct_n': int(np.sum(~svm_correct & indobert_correct)),
        'both_incorrect_n': int(np.sum(svm_error & indobert_error)),
        'hard_prediction_disagreement_n': int(np.sum(group['svm'].ne(group['indobert']))),
        'hard_prediction_disagreement_rate': float(group['svm'].ne(group['indobert']).mean()),
        'svm_error_rate': float(svm_error.mean()),
        'indobert_error_rate': float(indobert_error.mean()),
        'phi_error_correlation': phi_error_correlation,
    })
manual_base_model_contingency = pd.DataFrame(manual_complementarity_rows)
audit_reference = audit_for_evaluation.sort_values('row_id')
y_manual_reference = audit_reference['final_manual_label']
y_silver_reference = audit_reference['silver_label_lstm']
reference_weights = audit_reference['design_weight'].to_numpy(dtype=float)
silver_alignment = pd.DataFrame([{
    'comparison': 'LSTM silver label versus final manual label',
    'n_audit': int(len(audit_reference)),
    'sampling_design': '100 reviews per LSTM silver-label stratum',
    'accuracy_unweighted': float(accuracy_score(y_manual_reference, y_silver_reference)),
    'macro_f1_unweighted': float(f1_score(y_manual_reference, y_silver_reference, labels=LABEL_ORDER, average='macro', zero_division=0)),
    'accuracy_design_weighted': float(accuracy_score(y_manual_reference, y_silver_reference, sample_weight=reference_weights)),
    'macro_f1_design_weighted': float(f1_score(
        y_manual_reference, y_silver_reference, labels=LABEL_ORDER, average='macro',
        sample_weight=reference_weights, zero_division=0
    )),
}])

display(model_metrics_per_seed[[
    'seed', 'model', 'accuracy_unweighted', 'macro_f1_unweighted',
    'accuracy_design_weighted', 'macro_f1_design_weighted',
]])
display(model_metrics_summary)
display(silver_alignment)


### Write approved aggregate evaluation outputs

Exports only aggregate metrics by model and seed, aggregate confusion counts, and the silver-to-manual alignment summary. The report directory must already exist in the authorized environment. No row-level prediction, audit, candidate, or generated manifest file is written.

In [ ]:
aggregate_exports = {
    'human_label_final_model_metrics_per_seed.csv': model_metrics_per_seed,
    'human_label_final_model_metrics_summary.csv': model_metrics_summary,
    'human_label_final_model_confusion_matrices_per_seed.csv': pd.DataFrame(confusion_rows),
    'human_label_silver_alignment_final_300.csv': silver_alignment,
    'human_label_base_model_correctness_contingency_per_seed.csv': manual_base_model_contingency,
}

for filename, frame in aggregate_exports.items():
    frame.to_csv(FINAL_EVALUATION_REPORT_DIR / filename, index=False, encoding='utf-8-sig')
    print('Wrote aggregate evaluation file:', filename)

print('Manual-label evaluation exported as approved aggregate outputs only.')

### Create the silver-versus-manual confusion-matrix figure

Builds the aggregate confusion matrix for LSTM-derived silver labels versus final manual labels, preserving the figure cell because it supports the manuscript audit interpretation.


In [ ]:
from matplotlib import pyplot as plt

SILVER_MANUAL_MATRIX_PATH = FINAL_EVALUATION_REPORT_DIR / 'human_label_silver_vs_manual_confusion_matrix.csv'
SILVER_MANUAL_FIGURE_PATH = FIGURE_DIR / 'silver_vs_manual_confusion_matrix.png'

audit_for_matrix = human_audit.sort_values('locked_test_row').reset_index(drop=True)
silver_manual_matrix = confusion_matrix(
    audit_for_matrix['final_manual_label'], audit_for_matrix['silver_label_lstm'], labels=LABEL_ORDER
)
matrix_rows = []
for manual_index, manual_label in enumerate(LABEL_ORDER):
    for silver_index, silver_label in enumerate(LABEL_ORDER):
        matrix_rows.append({
            'true_manual_label': manual_label,
            'lstm_silver_label': silver_label,
            'n_reviews': int(silver_manual_matrix[manual_index, silver_index]),
        })
silver_manual_frame = pd.DataFrame(matrix_rows)
if int(silver_manual_frame['n_reviews'].sum()) != EXPECTED_AUDIT_SIZE:
    raise RuntimeError('The silver-versus-manual confusion matrix does not sum to 300 reviews.')
silver_manual_frame.to_csv(SILVER_MANUAL_MATRIX_PATH, index=False, encoding='utf-8-sig')

class_display = {'negatif': 'Negative', 'netral': 'Neutral', 'positif': 'Positive'}
fig, ax = plt.subplots(figsize=(6.6, 5.4), dpi=180)
image = ax.imshow(silver_manual_matrix, cmap='Blues')
fig.colorbar(image, ax=ax, label='Number of reviews')
ax.set_xticks(range(len(LABEL_ORDER)), [class_display[label] for label in LABEL_ORDER])
ax.set_yticks(range(len(LABEL_ORDER)), [class_display[label] for label in LABEL_ORDER])
ax.set_xlabel('LSTM-derived silver label')
ax.set_ylabel('Final manual label')
ax.set_title('LSTM Silver Labels Versus Final Manual Labels (n = 300)')
threshold = silver_manual_matrix.max() / 2
for row_index in range(len(LABEL_ORDER)):
    for column_index in range(len(LABEL_ORDER)):
        count = int(silver_manual_matrix[row_index, column_index])
        color = 'white' if count > threshold else 'black'
        ax.text(column_index, row_index, str(count), ha='center', va='center', color=color, fontsize=11)
ax.set_xticks(np.arange(-0.5, len(LABEL_ORDER), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(LABEL_ORDER), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=1.5)
ax.tick_params(which='minor', bottom=False, left=False)
fig.tight_layout()
fig.savefig(str(SILVER_MANUAL_FIGURE_PATH), dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)

display(silver_manual_frame.pivot(index='true_manual_label', columns='lstm_silver_label', values='n_reviews').reindex(
    index=LABEL_ORDER, columns=LABEL_ORDER
))
print('Silver-versus-manual matrix written to:', SILVER_MANUAL_MATRIX_PATH)
print('Silver-versus-manual figure written to:', SILVER_MANUAL_FIGURE_PATH)
